# Modeling (V15) — Two-Stage: Survival Ensemble + Close-Fire Timing Regression

**Core insight**: Within the 69 training close fires, all hit (event=1). This is pure regression
with no censoring. A dedicated timing model focused exclusively on close fires can learn hit-time
discriminators that get diluted in the 221-example survival analysis.

**Key data facts:**
- 89% of TEST close fires have zero velocity (closing_speed≈0, radial_growth≈0)
- Key correlates with hit time for zero-speed close fires: `num_perimeters_0_5h` (r=−0.35),
  `dt_first_last_0_5h` (r=−0.37), `low_temporal_resolution_0_5h` (r=+0.37), `alignment_abs` (r=−0.30)
- These features are dominated by the close/far distance signal in V8's 221-example survival model

**Architecture:**
1. V8 survival ensemble (RSF+GBM+Coxnet, all 221) → backbone blend
2. RidgeCV on 69 close fires → timing predictions → P(hit by h) via log-normal CDF
3. For close fires: `(1-beta)*survival + beta*timing`, beta found via OOF grid search
4. For far fires: survival only (unchanged)
5. V8-exact Platt@12h/@24h + isotonic@48h/@72h + cummax

**Fallback**: beta=0 → identical to V8 survival-only.


In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from scipy.optimize import minimize
from scipy.special import expit
from scipy.stats import norm as scipy_norm

from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression, RidgeCV
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

print('All imports OK')


All imports OK


In [2]:
SEED           = 42
N_SPLITS       = 10           # survival OOF folds (same as V8)
N_SPLITS_CLOSE = 7            # timing regression folds (69/7 ≈ 10 per fold)
DIST_THRESHOLD = 5_000
HORIZONS       = [12, 24, 48, 72]
MODEL_NAMES    = ['rsf', 'gbm', 'coxnet']
N_STARTS       = 11
MIN_COXNET_W   = 0.60
COXNET_ALPHA   = 0.05

# Sigma grid: controls smoothness of log-normal CDF conversion
# sigma too small → sharp step function (overfit); too large → flat probabilities (underfit)
SIGMA_GRID = [0.3, 0.5, 0.7, 1.0, 1.2, 1.5, 2.0]

# Beta grid: weight of timing model for close fires
# 0.0 = pure survival (V8), 0.5 = equal blend
BETA_GRID = [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

np.random.seed(SEED)

# V8's 42-feature set (identical)
BASE_FEATURES = [
    'num_perimeters_0_5h', 'dt_first_last_0_5h', 'low_temporal_resolution_0_5h',
    'area_first_ha', 'area_growth_rate_ha_per_h',
    'log1p_area_first', 'log1p_growth', 'radial_growth_rate_m_per_h',
    'area_growth_rel_0_5h', 'log_area_ratio_0_5h',
    'centroid_speed_m_per_h', 'spread_bearing_sin', 'spread_bearing_cos',
    'centroid_displacement_m',
    'closing_speed_m_per_h', 'closing_speed_abs_m_per_h',
    'projected_advance_m', 'dist_fit_r2_0_5h',
    'dist_accel_m_per_h2', 'dist_slope_ci_0_5h',
    'alignment_cos', 'alignment_abs',
]

ENGINEERED = [
    'log_dist_min', 'log_dist_sq', 'dist_close_5k', 'dist_close_30k',
    'tti_h', 'hazard_proxy',
    'is_dynamic', 'is_closing',
    'dynamic_x_dist', 'dist_x_low_res',
    'growth_momentum',
    'month_sin', 'month_cos', 'hour_sin', 'hour_cos',
    'dow_sin', 'dow_cos',
    'align_x_month_sin', 'bearing_x_month_cos', 'growth_x_dow_sin',
]

print(f'Config: SEED={SEED}, N_SPLITS={N_SPLITS}, N_SPLITS_CLOSE={N_SPLITS_CLOSE}')
print(f'Features: {len(BASE_FEATURES)+len(ENGINEERED)} (V8-exact)')
print(f'Sigma grid: {SIGMA_GRID}')
print(f'Beta grid:  {BETA_GRID}')


Config: SEED=42, N_SPLITS=10, N_SPLITS_CLOSE=7
Features: 42 (V8-exact)
Sigma grid: [0.3, 0.5, 0.7, 1.0, 1.2, 1.5, 2.0]
Beta grid:  [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]


In [3]:
os.chdir('/Users/marcelloborromeo/Downloads/BTT WiDS Datathon')


def engineer(df: pd.DataFrame) -> pd.DataFrame:
    """V8-exact feature engineering (42 features)."""
    df  = df.copy()
    eps = 1.0
    raw_dist = df['dist_min_ci_0_5h']

    df['log_dist_min']   = np.log1p(raw_dist)
    df['log_dist_sq']    = df['log_dist_min'] ** 2
    df['dist_close_5k']  = (raw_dist < 5_000).astype(float)
    df['dist_close_30k'] = (raw_dist < 30_000).astype(float)

    closing = df['closing_speed_m_per_h'].clip(lower=0)
    df['tti_h']        = (raw_dist / (closing + eps)).clip(upper=500)
    df['hazard_proxy'] = df['alignment_abs'] * closing / (raw_dist + eps)

    df['is_dynamic']     = (df['area_growth_rate_ha_per_h'] > 0).astype(float)
    df['is_closing']     = (df['closing_speed_m_per_h']     > 0).astype(float)
    df['dynamic_x_dist'] = df['is_dynamic']   * df['log_dist_min']
    df['dist_x_low_res'] = df['log_dist_min'] * df['low_temporal_resolution_0_5h']
    df['growth_momentum']= df['log1p_area_first'] * df['radial_growth_rate_m_per_h']

    df['month_sin'] = np.sin(2 * np.pi * df['event_start_month']     / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['event_start_month']     / 12)
    df['hour_sin']  = np.sin(2 * np.pi * df['event_start_hour']      / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['event_start_hour']      / 24)
    df['dow_sin']   = np.sin(2 * np.pi * df['event_start_dayofweek'] / 7)
    df['dow_cos']   = np.cos(2 * np.pi * df['event_start_dayofweek'] / 7)

    df['align_x_month_sin']   = df['alignment_abs']      * df['month_sin']
    df['bearing_x_month_cos'] = df['spread_bearing_cos'] * df['month_cos']
    df['growth_x_dow_sin']    = df['log1p_growth']        * df['dow_sin']

    return df


train    = pd.read_csv('train.csv')
test     = pd.read_csv('test.csv')
train_fe = engineer(train)
test_fe  = engineer(test)

FEAT_COLS = [f for f in BASE_FEATURES + ENGINEERED if f in train_fe.columns]
print(f'Total features: {len(FEAT_COLS)}  (expected 42)')

y_event = train['event'].values.astype(bool)
y_time  = train['time_to_hit_hours'].values.astype(float)
y_surv  = Surv.from_arrays(y_event, y_time)

close_train = train['dist_min_ci_0_5h'].values < DIST_THRESHOLD
close_test  = test['dist_min_ci_0_5h'].values  < DIST_THRESHOLD

preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])
X_train = preprocessor.fit_transform(train_fe[FEAT_COLS].values)
X_test  = preprocessor.transform(test_fe[FEAT_COLS].values)

# Close-fire subsets for timing regression
X_close      = X_train[close_train]              # [69, 42]
y_time_close = y_time[close_train]               # hit times (all observed, no censoring)
y_log_close  = np.log1p(y_time_close)            # log1p scale for regression target

print(f'Train: {X_train.shape}  events={y_event.sum()}  close={close_train.sum()}')
print(f'Test:  {X_test.shape}   close={close_test.sum()}')
print(f'Close train: {X_close.shape}  log-time range [{y_log_close.min():.2f}, {y_log_close.max():.2f}]')
print()
print(f'Close test zero-speed: {((test[close_test]["closing_speed_m_per_h"].abs() < 1) & (test[close_test]["radial_growth_rate_m_per_h"] < 0.1)).sum()} / {close_test.sum()} ({((test[close_test]["closing_speed_m_per_h"].abs() < 1) & (test[close_test]["radial_growth_rate_m_per_h"] < 0.1)).sum()/max(close_test.sum(),1)*100:.0f}%)')


Total features: 42  (expected 42)
Train: (221, 42)  events=69  close=69
Test:  (95, 42)   close=28
Close train: (69, 42)  log-time range [0.00, 4.22]

Close test zero-speed: 25 / 28 (89%)


In [4]:
def surv_fn_to_probs(surv_fns) -> np.ndarray:
    out = np.zeros((len(surv_fns), len(HORIZONS)))
    for i, sf in enumerate(surv_fns):
        t_min, t_max = sf.domain
        for j, t in enumerate(HORIZONS):
            out[i, j] = 1.0 - float(sf(float(np.clip(t, t_min, t_max))))
    return out


def check_monotonicity(p: np.ndarray, label: str = '') -> int:
    v_12_24 = int((p[:, 0] > p[:, 1]).sum())
    v_24_48 = int((p[:, 1] > p[:, 2]).sum())
    v_48_72 = int((p[:, 2] > p[:, 3]).sum())
    total   = v_12_24 + v_24_48 + v_48_72
    tag = f' [{label}]' if label else ''
    print(f'  Monotonicity{tag}: @12>@24={v_12_24}  @24>@48={v_24_48}  @48>@72={v_48_72}  total={total}')
    return total


def brier_score_at(p_h, y_evt, y_t, h):
    hit  = (y_evt == 1) & (y_t <= h)
    excl = (y_evt == 0) & (y_t <  h)
    keep = ~excl
    if keep.sum() == 0:
        return np.nan
    return float(np.mean((p_h[keep] - hit[keep].astype(float)) ** 2))


def weighted_brier(probs, y_evt, y_t):
    bs24 = brier_score_at(probs[:, 1], y_evt, y_t, 24)
    bs48 = brier_score_at(probs[:, 2], y_evt, y_t, 48)
    bs72 = brier_score_at(probs[:, 3], y_evt, y_t, 72)
    return 0.3 * bs24 + 0.4 * bs48 + 0.3 * bs72


def hybrid_score(probs, y_evt, y_t, risk_scores=None):
    wbs  = weighted_brier(probs, y_evt, y_t)
    bs24 = brier_score_at(probs[:, 1], y_evt, y_t, 24)
    bs48 = brier_score_at(probs[:, 2], y_evt, y_t, 48)
    bs72 = brier_score_at(probs[:, 3], y_evt, y_t, 72)
    risk = risk_scores if risk_scores is not None else probs[:, 2]
    try:
        c, *_ = concordance_index_censored(y_evt.astype(bool), y_t, risk)
    except Exception:
        c = 0.5
    return 0.3 * c + 0.7 * (1 - wbs), c, wbs, bs24, bs48, bs72


def platt_fit(raw_oof_h, y_evt, y_t, horizon):
    hit  = (y_evt == 1) & (y_t <= horizon)
    excl = (y_evt == 0) & (y_t <  horizon)
    keep = ~excl
    lr   = LogisticRegression(C=1e6, solver='lbfgs', max_iter=2000, fit_intercept=True)
    lr.fit(raw_oof_h[keep].reshape(-1, 1), hit[keep].astype(int))
    return float(lr.coef_[0, 0]), float(lr.intercept_[0])


def iso_fit(raw_oof_h, y_evt, y_t, horizon):
    hit  = (y_evt == 1) & (y_t <= horizon)
    excl = (y_evt == 0) & (y_t <  horizon)
    keep = ~excl
    iso  = IsotonicRegression(out_of_bounds='clip')
    iso.fit(raw_oof_h[keep], hit[keep].astype(float))
    return iso


def apply_calibration(raw_blend, platts, iso_48, iso_72, postproc=True):
    a12, b12 = platts[12]
    a24, b24 = platts[24]
    p12 = expit(a12 * raw_blend[:, 0] + b12)
    p24 = expit(a24 * raw_blend[:, 1] + b24)
    p48 = iso_48.predict(raw_blend[:, 2])
    p72 = iso_72.predict(raw_blend[:, 3])
    out = np.column_stack([p12, p24, p48, p72])
    if postproc:
        out = np.clip(np.maximum.accumulate(out, axis=1), 0.0, 1.0)
    return out


def timing_to_probs(log1p_pred, sigma):
    """Convert log1p(time) predictions to P(hit by h) via log-normal CDF.

    P(hit by h) = Φ((log1p(h) - predicted_log1p_time) / sigma)

    Automatically monotone in h (CDF is non-decreasing).
    sigma controls smoothness: small → sharp transition, large → smooth.
    """
    probs = np.zeros((len(log1p_pred), len(HORIZONS)))
    for j, h in enumerate(HORIZONS):
        probs[:, j] = scipy_norm.cdf((np.log1p(h) - log1p_pred) / sigma)
    return np.clip(np.maximum.accumulate(probs, axis=1), 0.0, 1.0)


print('All utility functions defined (including timing_to_probs).')


All utility functions defined (including timing_to_probs).


In [5]:
def build_rsf(n_estimators=200):
    return RandomSurvivalForest(
        n_estimators=n_estimators, max_depth=4, min_samples_leaf=10,
        max_features='sqrt', n_jobs=-1, random_state=SEED, oob_score=True,
    )


def build_gbm():
    return GradientBoostingSurvivalAnalysis(
        n_estimators=150, learning_rate=0.04, max_depth=2,
        subsample=0.8, min_samples_leaf=15, random_state=SEED,
    )


def build_coxnet(alpha=COXNET_ALPHA):
    return CoxnetSurvivalAnalysis(
        alphas=[alpha], l1_ratio=0.5,
        fit_baseline_model=True, max_iter=10_000,
    )


def build_ridge():
    """RidgeCV for close-fire timing regression.

    Auto-selects alpha from grid. L2 penalty prevents overfitting n=69 examples.
    """
    return RidgeCV(alphas=[0.1, 1, 10, 50, 100, 500, 1000])


print(f'Model builders ready: {MODEL_NAMES} + Ridge (timing)')


Model builders ready: ['rsf', 'gbm', 'coxnet'] + Ridge (timing)


In [6]:
n   = len(X_train)
oof = {name: np.zeros((n, len(HORIZONS))) for name in MODEL_NAMES}

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
print(f'Stage 1: Survival OOF ({N_SPLITS}-fold, {len(FEAT_COLS)} features)...')

for fold, (tr_i, va_i) in enumerate(skf.split(X_train, y_event.astype(int))):
    X_tr, X_va = X_train[tr_i], X_train[va_i]
    y_tr        = y_surv[tr_i]
    print(f'  Fold {fold+1:2d}/{N_SPLITS}  (train={len(tr_i)}, val={len(va_i)})', end=' ')

    m = build_rsf(); m.fit(X_tr, y_tr)
    oof['rsf'][va_i] = surv_fn_to_probs(m.predict_survival_function(X_va))
    print('RSF✓', end=' ')

    m = build_gbm(); m.fit(X_tr, y_tr)
    oof['gbm'][va_i] = surv_fn_to_probs(m.predict_survival_function(X_va))
    print('GBM✓', end=' ')

    try:
        m = build_coxnet(); m.fit(X_tr, y_tr)
        oof['coxnet'][va_i] = surv_fn_to_probs(m.predict_survival_function(X_va))
        print('Coxnet✓')
    except Exception as e:
        print(f'Coxnet[fallback→RSF: {str(e)[:30]}]')
        oof['coxnet'][va_i] = oof['rsf'][va_i].copy()

print()
print('Survival OOF done. Close-fire @48h per model:')
for name in MODEL_NAMES:
    vals = oof[name][close_train, 2]
    print(f'  {name:<8s}  mean={vals.mean():.4f}  std={vals.std():.4f}')


Stage 1: Survival OOF (10-fold, 42 features)...
  Fold  1/10  (train=198, val=23) 

RSF✓ GBM✓ Coxnet✓
  Fold  2/10  (train=199, val=22) 

RSF✓ GBM✓ Coxnet✓
  Fold  3/10  (train=199, val=22) 

RSF✓ GBM✓ Coxnet✓
  Fold  4/10  (train=199, val=22) 

RSF✓ GBM✓ Coxnet✓
  Fold  5/10  (train=199, val=22) 

...

RSF✓ GBM✓ Coxnet✓
  Fold  7/10  (train=199, val=22) 

RSF✓ GBM✓ Coxnet✓
  Fold  8/10  (train=199, val=22) 

RSF✓ GBM✓ Coxnet✓
  Fold  9/10  (train=199, val=22) 

RSF✓ GBM✓ Coxnet✓
  Fold 10/10  (train=199, val=22) 

RSF✓ GBM✓ Coxnet✓

Survival OOF done. Close-fire @48h per model:
  rsf       mean=0.8598  std=0.0325
  gbm       mean=0.9167  std=0.0850
  coxnet    mean=0.9337  std=0.0766


In [7]:
def make_hybrid_obj(oof_dict, names, y_ev, y_t):
    plist = [oof_dict[n] for n in names]
    n_m   = len(names)

    def obj(w_raw):
        w = np.abs(w_raw); s = w.sum()
        if s < 1e-9: return 1.0
        w = w / s
        blend = sum(w[i] * plist[i] for i in range(n_m))
        bs24 = brier_score_at(blend[:, 1], y_ev, y_t, 24)
        bs48 = brier_score_at(blend[:, 2], y_ev, y_t, 48)
        bs72 = brier_score_at(blend[:, 3], y_ev, y_t, 72)
        wbs  = 0.3 * bs24 + 0.4 * bs48 + 0.3 * bs72
        try:
            c, *_ = concordance_index_censored(y_ev.astype(bool), y_t, blend[:, 2])
        except Exception:
            c = 0.5
        return -(0.3 * c + 0.7 * (1.0 - wbs))
    return obj


obj         = make_hybrid_obj(oof, MODEL_NAMES, y_event, y_time)
n_m         = len(MODEL_NAMES)
coxnet_idx  = MODEL_NAMES.index('coxnet')
rng         = np.random.RandomState(SEED + 1)

starts = [np.ones(n_m) / n_m]
for _ in range(9):
    starts.append(rng.dirichlet(np.ones(n_m)))
v8_like = np.zeros(n_m)
v8_like[coxnet_idx]              = 0.79
v8_like[MODEL_NAMES.index('gbm')] = 0.21
starts.append(v8_like)

print(f'Optimising survival blend weights ({N_STARTS} starts)...')
best_obj_con = np.inf; best_w_con = None
best_obj_all = np.inf; best_w_all = None

for w0 in starts:
    res = minimize(obj, w0, method='Nelder-Mead',
                   options={'maxiter': 5000, 'xatol': 1e-7, 'fatol': 1e-8})
    w_n = np.abs(res.x); w_n /= (w_n.sum() + 1e-12)
    val = res.fun
    if val < best_obj_all: best_obj_all = val; best_w_all = w_n.copy()
    if w_n[coxnet_idx] >= MIN_COXNET_W and val < best_obj_con:
        best_obj_con = val; best_w_con = w_n.copy()

best_w = best_w_con if best_w_con is not None else best_w_all
weights = dict(zip(MODEL_NAMES, best_w))
print('Survival blend weights:')
for name, w in zip(MODEL_NAMES, best_w):
    print(f'  {name:<8s}: {w:.4f}  {"#" * int(round(w * 30))}')

raw_blend_oof = sum(weights[n] * oof[n] for n in MODEL_NAMES)
print(f'\nRaw blend @48h close: mean={raw_blend_oof[close_train, 2].mean():.4f}  std={raw_blend_oof[close_train, 2].std():.4f}')


Optimising survival blend weights (11 starts)...


Survival blend weights:
  rsf     : 0.0442  #
  gbm     : 0.1776  #####
  coxnet  : 0.7782  #######################

Raw blend @48h close: mean=0.9274  std=0.0741


In [8]:
print('Stage 2: Close-fire timing regression...')
print(f'  Data: {X_close.shape[0]} close fires (all event=1, no censoring)')
print(f'  Target: log1p(time_to_hit_hours)  range [{y_log_close.min():.2f}, {y_log_close.max():.2f}]')
print(f'  Model: RidgeCV  |  CV: {N_SPLITS_CLOSE}-fold KFold')
print()

timing_oof = np.zeros(len(X_close))  # predicted log1p-times for close fires
timing_alphas = []

kf_close = KFold(n_splits=N_SPLITS_CLOSE, shuffle=True, random_state=SEED)

for fold, (tr_i, va_i) in enumerate(kf_close.split(X_close)):
    ridge = build_ridge()
    ridge.fit(X_close[tr_i], y_log_close[tr_i])
    timing_oof[va_i] = ridge.predict(X_close[va_i])
    timing_alphas.append(ridge.alpha_)
    print(f'  Fold {fold+1}/{N_SPLITS_CLOSE}: alpha={ridge.alpha_:.1f}  '
          f'val RMSE={np.sqrt(np.mean((timing_oof[va_i] - y_log_close[va_i])**2)):.4f}')

# OOF quality
oof_rmse = np.sqrt(np.mean((timing_oof - y_log_close)**2))
oof_r2   = np.corrcoef(timing_oof, y_log_close)[0, 1]**2
print(f'\nTiming OOF: RMSE={oof_rmse:.4f}  R²={oof_r2:.4f}  alpha_mean={np.mean(timing_alphas):.1f}')
print()

# Show worst predictions (late-hitters)
oof_err = np.abs(timing_oof - y_log_close)
worst   = np.argsort(oof_err)[-5:]
print('Worst 5 timing predictions:')
for idx in worst:
    print(f'  idx={idx:3d}  pred={np.expm1(timing_oof[idx]):6.1f}h  actual={y_time_close[idx]:6.1f}h  err={oof_err[idx]:.3f}')


Stage 2: Close-fire timing regression...
  Data: 69 close fires (all event=1, no censoring)
  Target: log1p(time_to_hit_hours)  range [0.00, 4.22]
  Model: RidgeCV  |  CV: 7-fold KFold

  Fold 1/7: alpha=50.0  val RMSE=1.0343
  Fold 2/7: alpha=100.0  val RMSE=1.0733
  Fold 3/7: alpha=50.0  val RMSE=0.5533
  Fold 4/7: alpha=50.0  val RMSE=0.9340
  Fold 5/7: alpha=100.0  val RMSE=1.4041
  Fold 6/7: alpha=100.0  val RMSE=0.6390
  Fold 7/7: alpha=50.0  val RMSE=0.9170

Timing OOF: RMSE=0.9734  R²=0.2903  alpha_mean=71.4

Worst 5 timing predictions:
  idx=  0  pred=   2.8h  actual=  22.0h  err=1.808
  idx= 44  pred=   4.5h  actual=  45.3h  err=2.131
  idx= 41  pred=   9.1h  actual=   0.1h  err=2.184
  idx= 63  pred=   9.0h  actual=   0.0h  err=2.294
  idx= 31  pred=   2.5h  actual=  43.8h  err=2.549


In [9]:
# Calibrate sigma: controls smoothness of log-normal CDF conversion
# Minimise WBS on 69 close-fire OOF predictions
# sigma too small → sharp step function; too large → flat probabilities

print('=== Sigma Calibration (minimise close-fire WBS) ===')
best_sigma = SIGMA_GRID[0]
best_wbs_close = np.inf

y_evt_c = y_event[close_train]
y_t_c   = y_time_close

for sigma in SIGMA_GRID:
    probs_c = timing_to_probs(timing_oof, sigma)
    bs24_c = brier_score_at(probs_c[:, 1], y_evt_c, y_t_c, 24)
    bs48_c = brier_score_at(probs_c[:, 2], y_evt_c, y_t_c, 48)
    bs72_c = brier_score_at(probs_c[:, 3], y_evt_c, y_t_c, 72)
    wbs_c  = 0.3 * bs24_c + 0.4 * bs48_c + 0.3 * bs72_c
    try:
        c_c, *_ = concordance_index_censored(y_evt_c.astype(bool), y_t_c, probs_c[:, 2])
    except Exception:
        c_c = 0.5
    h_c = 0.3 * c_c + 0.7 * (1 - wbs_c)
    print(f'  sigma={sigma:.1f}: C_close={c_c:.4f}  WBS_close={wbs_c:.4f}  Hybrid_close={h_c:.4f}',
          end='')
    if wbs_c < best_wbs_close:
        best_wbs_close = wbs_c
        best_sigma = sigma
        print('  ← best WBS')
    else:
        print()

print(f'\nBest sigma: {best_sigma}  (WBS_close={best_wbs_close:.4f})')
timing_probs_close = timing_to_probs(timing_oof, best_sigma)  # [69, 4]
print(f'Timing probs close (sigma={best_sigma}):')
print(f'  @12h: mean={timing_probs_close[:,0].mean():.4f}')
print(f'  @24h: mean={timing_probs_close[:,1].mean():.4f}')
print(f'  @48h: mean={timing_probs_close[:,2].mean():.4f}  std={timing_probs_close[:,2].std():.4f}')
print(f'  @72h: mean={timing_probs_close[:,3].mean():.4f}')
print()
print('Survival OOF for close fires (compare):')
print(f'  @48h: mean={raw_blend_oof[close_train,2].mean():.4f}  std={raw_blend_oof[close_train,2].std():.4f}')


=== Sigma Calibration (minimise close-fire WBS) ===
  sigma=0.3: C_close=0.6194  WBS_close=0.0434  Hybrid_close=0.8554  ← best WBS
  sigma=0.5: C_close=0.6971  WBS_close=0.0421  Hybrid_close=0.8797  ← best WBS
  sigma=0.7: C_close=0.6931  WBS_close=0.0403  Hybrid_close=0.8797  ← best WBS
  sigma=1.0: C_close=0.6931  WBS_close=0.0393  Hybrid_close=0.8804  ← best WBS
  sigma=1.2: C_close=0.6931  WBS_close=0.0402  Hybrid_close=0.8798
  sigma=1.5: C_close=0.6931  WBS_close=0.0441  Hybrid_close=0.8770
  sigma=2.0: C_close=0.6931  WBS_close=0.0553  Hybrid_close=0.8692

Best sigma: 1.0  (WBS_close=0.0393)
Timing probs close (sigma=1.0):
  @12h: mean=0.7555
  @24h: mean=0.8959
  @48h: mean=0.9682  std=0.0335
  @72h: mean=0.9866

Survival OOF for close fires (compare):
  @48h: mean=0.9274  std=0.0741


In [10]:
# Grid search over beta: weight of timing model for close fires
# For each beta: mix close-fire predictions, fit V8-exact calibration, compute hybrid OOF
# This is done on TRAINING OOF only — no test leakage

print('=== Beta Search: blend survival OOF + timing model for close fires ===')
print(f'Grid: {BETA_GRID}')
print()

best_beta    = 0.0
best_hybrid  = -np.inf
best_cal_oof = None
best_cals    = None
beta_results = []

for beta in BETA_GRID:
    # Mix close-fire predictions
    mixed_oof = raw_blend_oof.copy()
    mixed_oof[close_train] = (1 - beta) * raw_blend_oof[close_train] + beta * timing_probs_close

    # Fit V8-exact calibration on the mixed OOF
    platts_b = {}
    platts_b[12] = platt_fit(mixed_oof[:, 0], y_event, y_time, 12)
    platts_b[24] = platt_fit(mixed_oof[:, 1], y_event, y_time, 24)
    iso48_b      = iso_fit(mixed_oof[:, 2], y_event, y_time, 48)
    iso72_b      = iso_fit(mixed_oof[:, 3], y_event, y_time, 72)

    cal_b = apply_calibration(mixed_oof, platts_b, iso48_b, iso72_b)
    h, c, wbs, bs24, bs48, bs72 = hybrid_score(
        cal_b, y_event, y_time, risk_scores=mixed_oof[:, 2]
    )
    beta_results.append((beta, h, c, wbs))
    print(f'  beta={beta:.2f}: Hybrid={h:.4f}  C={c:.4f}  WBS={wbs:.4f}  BS@48={bs48:.4f}',
          end='')

    if h > best_hybrid:
        best_hybrid  = h
        best_beta    = beta
        best_cal_oof = cal_b
        best_cals    = {'platts': platts_b, 'iso48': iso48_b, 'iso72': iso72_b}
        print('  ← best')
    else:
        print()

print()
print(f'Best beta: {best_beta:.2f}  Best OOF Hybrid: {best_hybrid:.4f}')
print(f'  vs V8 Hybrid:  0.9717  delta={best_hybrid - 0.9717:+.4f}')
print(f'  vs V14 Hybrid: 0.9713  delta={best_hybrid - 0.9713:+.4f}')


=== Beta Search: blend survival OOF + timing model for close fires ===
Grid: [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]

  beta=0.00: Hybrid=0.9719  C=0.9410  WBS=0.0148  BS@48=0.0148  ← best
  beta=0.05: Hybrid=0.9719  C=0.9409  WBS=0.0147  BS@48=0.0148
  beta=0.10: Hybrid=0.9719  C=0.9405  WBS=0.0147  BS@48=0.0148
  beta=0.15: Hybrid=0.9719  C=0.9406  WBS=0.0146  BS@48=0.0147
  beta=0.20: Hybrid=0.9721  C=0.9410  WBS=0.0146  BS@48=0.0147  ← best
  beta=0.25: Hybrid=0.9721  C=0.9411  WBS=0.0146  BS@48=0.0147  ← best
  beta=0.30: Hybrid=0.9721  C=0.9409  WBS=0.0145  BS@48=0.0148
  beta=0.35: Hybrid=0.9721  C=0.9410  WBS=0.0145  BS@48=0.0148
  beta=0.40: Hybrid=0.9722  C=0.9412  WBS=0.0145  BS@48=0.0147  ← best
  beta=0.45: Hybrid=0.9722  C=0.9410  WBS=0.0144  BS@48=0.0147
  beta=0.50: Hybrid=0.9721  C=0.9407  WBS=0.0144  BS@48=0.0147

Best beta: 0.40  Best OOF Hybrid: 0.9722
  vs V8 Hybrid:  0.9717  delta=+0.0005
  vs V14 Hybrid: 0.9713  delta=+0.0009


In [11]:
h15, c15, wbs15, bs24_15, bs48_15, bs72_15 = hybrid_score(
    best_cal_oof, y_event, y_time, risk_scores=
    (raw_blend_oof.copy().__setitem__(slice(None), raw_blend_oof.copy()) or
     (lambda m: m)(raw_blend_oof.copy()))[:, 2]
)

# Recompute cleanly
mixed_best = raw_blend_oof.copy()
mixed_best[close_train] = (1 - best_beta) * raw_blend_oof[close_train] + best_beta * timing_probs_close
h15, c15, wbs15, bs24_15, bs48_15, bs72_15 = hybrid_score(
    best_cal_oof, y_event, y_time, risk_scores=mixed_best[:, 2]
)

print('=== V15 OOF Metrics ===')
print(f'  C-index       : {c15:.4f}  (V8=0.9403, V14=0.9391)')
print(f'  Brier@24h     : {bs24_15:.4f}  (V8=0.0296, V14=0.0297)')
print(f'  Brier@48h     : {bs48_15:.4f}  (V8=0.0149, V14=0.0148)')
print(f'  Weighted Brier: {wbs15:.4f}  (V8=0.0148, V14=0.0148)')
print(f'  Hybrid        : {h15:.4f}  (V8=0.9717, V14=0.9713)')
print()
print(f'  best_beta={best_beta:.2f}  best_sigma={best_sigma:.1f}')
print()

print('OOF monotonicity:')
check_monotonicity(best_cal_oof, 'V15 OOF')
print()

print('Close-fire OOF stats (calibrated):')
print(f'  @48h: mean={best_cal_oof[close_train, 2].mean():.4f}  std={best_cal_oof[close_train, 2].std():.4f}')
print(f'  @48h V8 had: std≈0.0784  (V13 had: std≈0.015)')


=== V15 OOF Metrics ===
  C-index       : 0.9412  (V8=0.9403, V14=0.9391)
  Brier@24h     : 0.0285  (V8=0.0296, V14=0.0297)
  Brier@48h     : 0.0147  (V8=0.0149, V14=0.0148)
  Weighted Brier: 0.0145  (V8=0.0148, V14=0.0148)
  Hybrid        : 0.9722  (V8=0.9717, V14=0.9713)

  best_beta=0.40  best_sigma=1.0

OOF monotonicity:
  Monotonicity [V15 OOF]: @12>@24=0  @24>@48=0  @48>@72=0  total=0

Close-fire OOF stats (calibrated):
  @48h: mean=0.9657  std=0.0808
  @48h V8 had: std≈0.0784  (V13 had: std≈0.015)


In [12]:
versions = {
    'V8  (LB=0.97090)': {'c_idx': 0.9403, 'bs24': 0.0296, 'bs48': 0.0149, 'wbs': 0.0148, 'hybrid': 0.9717},
    'V11 (LB=0.97056)': {'c_idx': 0.9388, 'bs24': 0.0283, 'bs48': 0.0146, 'wbs': 0.0143, 'hybrid': 0.9716},
    'V13 (LB=0.95962)': {'c_idx': 0.9298, 'bs24': 0.0320, 'bs48': 0.0180, 'wbs': 0.0167, 'hybrid': 0.9673},
    'V14 (OOF=0.9713)': {'c_idx': 0.9391, 'bs24': 0.0297, 'bs48': 0.0148, 'wbs': 0.0148, 'hybrid': 0.9713},
    'V15 (this)':        {'c_idx': c15,    'bs24': bs24_15,'bs48': bs48_15,'wbs': wbs15,  'hybrid': h15},
}

cols       = ['c_idx', 'bs24', 'bs48', 'wbs', 'hybrid']
col_labels = ['C-index', 'Brier@24h', 'Brier@48h', 'Wtd Brier', 'Hybrid']
w = 20

print('=== OOF Score Comparison ===')
print()
print(f"  {'Metric':<20s}" + ''.join(f'{k:>{w}s}' for k in versions))
print('-' * (20 + w * len(versions)))
for ck, cl in zip(cols, col_labels):
    row = f'  {cl:<20s}' + ''.join(f'{v[ck]:>{w}.4f}' for v in versions.values())
    print(row)

print()
print(f'V15 vs V8  Hybrid: {h15 - 0.9717:+.4f}')
print(f'V15 vs V8  C-idx : {c15 - 0.9403:+.4f}')
print(f'V15 vs V8  WBS   : {wbs15 - 0.0148:+.4f}  (negative = better)')


=== OOF Score Comparison ===

  Metric                  V8  (LB=0.97090)    V11 (LB=0.97056)    V13 (LB=0.95962)    V14 (OOF=0.9713)          V15 (this)
------------------------------------------------------------------------------------------------------------------------
  C-index                           0.9403              0.9388              0.9298              0.9391              0.9412
  Brier@24h                         0.0296              0.0283              0.0320              0.0297              0.0285
  Brier@48h                         0.0149              0.0146              0.0180              0.0148              0.0147
  Wtd Brier                         0.0148              0.0143              0.0167              0.0148              0.0145
  Hybrid                            0.9717              0.9716              0.9673              0.9713              0.9722

V15 vs V8  Hybrid: +0.0005
V15 vs V8  C-idx : +0.0009
V15 vs V8  WBS   : -0.0003  (negative = better)


In [13]:
print('Training final models...')
final_models = {}

# Survival models (Stage 1 final — higher n_estimators)
m = build_rsf(n_estimators=400)
m.fit(X_train, y_surv)
final_models['rsf'] = m
print(f'  RSF done  (OOB={m.oob_score_:.4f})')

m = build_gbm()
m.fit(X_train, y_surv)
final_models['gbm'] = m
print('  GBM done')

m = build_coxnet()
m.fit(X_train, y_surv)
final_models['coxnet'] = m
print('  Coxnet done')

# Timing regression (Stage 2 final — all 69 close fires)
final_ridge = build_ridge()
final_ridge.fit(X_close, y_log_close)
print(f'  Ridge done  (alpha={final_ridge.alpha_:.1f})')
print()
print('Final training complete.')


Training final models...


  RSF done  (OOB=0.9387)
  GBM done
  Coxnet done
  Ridge done  (alpha=50.0)

Final training complete.


In [14]:
print('Generating test predictions...')

# Stage 1: Survival blend for ALL test fires
test_raw_preds = {}
for name in MODEL_NAMES:
    test_raw_preds[name] = surv_fn_to_probs(
        final_models[name].predict_survival_function(X_test)
    )
test_raw = sum(weights[n] * test_raw_preds[n] for n in MODEL_NAMES)

# Stage 2: Timing model for CLOSE test fires
X_test_close   = X_test[close_test]
timing_test_log = final_ridge.predict(X_test_close)
timing_test_probs = timing_to_probs(timing_test_log, best_sigma)

# Stage 4: Beta blend for close test fires
test_mixed = test_raw.copy()
if best_beta > 0:
    test_mixed[close_test] = (
        (1 - best_beta) * test_raw[close_test] +
        best_beta        * timing_test_probs
    )

# Stage 5: Apply best calibration
platts_best = best_cals['platts']
iso48_best  = best_cals['iso48']
iso72_best  = best_cals['iso72']
test_cal    = apply_calibration(test_mixed, platts_best, iso48_best, iso72_best)

print(f'Test predictions (beta={best_beta:.2f}, sigma={best_sigma:.1f}):')
print(f'  Close @12h: {test_cal[close_test, 0].mean():.4f}')
print(f'  Close @24h: {test_cal[close_test, 1].mean():.4f}')
print(f'  Close @48h: {test_cal[close_test, 2].mean():.4f}  std={test_cal[close_test, 2].std():.4f}')
print(f'  Far   @48h: {test_cal[~close_test, 2].mean():.4f}')
print(f'  All   @72h: {test_cal[:, 3].mean():.4f}  (should = 1.00)')
print(f'  Unique @48h values: {len(np.unique(test_cal[:, 2].round(4)))}/95')

# Timing prediction sanity for close test fires
print()
print(f'Close test timing predictions:')
print(f'  log1p-time: mean={timing_test_log.mean():.3f}  std={timing_test_log.std():.3f}')
print(f'  Actual time (expm1): mean={np.expm1(timing_test_log).mean():.1f}h  min={np.expm1(timing_test_log).min():.1f}h  max={np.expm1(timing_test_log).max():.1f}h')


Generating test predictions...
Test predictions (beta=0.40, sigma=1.0):
  Close @12h: 0.6522
  Close @24h: 0.9010
  Close @48h: 0.9615  std=0.0836
  Far   @48h: 0.0060
  All   @72h: 1.0000  (should = 1.00)
  Unique @48h values: 26/95

Close test timing predictions:
  log1p-time: mean=1.859  std=0.614
  Actual time (expm1): mean=6.5h  min=0.7h  max=15.7h


In [15]:
print('=== Submission Monotonicity Check ===')
v_12_24 = int((test_cal[:, 0] > test_cal[:, 1]).sum())
v_24_48 = int((test_cal[:, 1] > test_cal[:, 2]).sum())
v_48_72 = int((test_cal[:, 2] > test_cal[:, 3]).sum())
total   = v_12_24 + v_24_48 + v_48_72

print(f'  @12h > @24h: {v_12_24}')
print(f'  @24h > @48h: {v_24_48}')
print(f'  @48h > @72h: {v_48_72}')
print(f'  Total      : {total}')
assert total == 0, f'SUBMISSION BLOCKED: {total} monotonicity violations!'
print('  PASSED — safe to submit')


=== Submission Monotonicity Check ===
  @12h > @24h: 0
  @24h > @48h: 0
  @48h > @72h: 0
  Total      : 0
  PASSED — safe to submit


In [16]:
out_dir = Path('Artifacts and Submissions')
out_dir.mkdir(exist_ok=True)

sub = pd.DataFrame({
    'event_id': test['event_id'],
    'prob_12h':  test_cal[:, 0],
    'prob_24h':  test_cal[:, 1],
    'prob_48h':  test_cal[:, 2],
    'prob_72h':  test_cal[:, 3],
})

sub_path = out_dir / 'submission (V15).csv'
sub.to_csv(sub_path, index=False)
print(f'Submission saved: {sub_path}  shape={sub.shape}')
print()
print(sub[['prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']].describe().round(4))


Submission saved: Artifacts and Submissions/submission (V15).csv  shape=(95, 5)

       prob_12h  prob_24h  prob_48h  prob_72h
count   95.0000   95.0000   95.0000      95.0
mean     0.1964    0.2698    0.2877       1.0
std      0.3229    0.4128    0.4403       0.0
min      0.0052    0.0052    0.0052       1.0
25%      0.0054    0.0054    0.0054       1.0
50%      0.0058    0.0058    0.0058       1.0
75%      0.3561    0.8121    0.8121       1.0
max      0.9725    0.9765    1.0000       1.0


In [17]:
artifacts = {
    'weights':      weights,
    'best_beta':    best_beta,
    'best_sigma':   best_sigma,
    'calibrators':  best_cals,
    'preprocessor': preprocessor,
    'feat_cols':    FEAT_COLS,
    'horizons':     HORIZONS,
    'final_models': final_models,
    'final_ridge':  final_ridge,
    'oof_scores': {
        'c_idx': c15, 'wbs': wbs15, 'hybrid': h15,
        'bs24': bs24_15, 'bs48': bs48_15, 'bs72': bs72_15,
    },
    'timing_oof_r2':  float(np.corrcoef(timing_oof, y_log_close)[0, 1]**2),
    'test_close_48h': float(test_cal[close_test, 2].mean()),
}

art_path = out_dir / 'model_artifacts (V15).pkl'
with open(art_path, 'wb') as f:
    pickle.dump(artifacts, f)
print(f'Artifacts saved: {art_path}')

print()
print('=== V15 Final Summary ===')
print(f'  Architecture : Survival ensemble (Stage 1) + Ridge timing (Stage 2)')
print(f'  Close fires  : beta={best_beta:.2f}*timing + {1-best_beta:.2f}*survival')
print(f'  Far fires    : survival only')
print(f'  Sigma        : {best_sigma:.1f}  (log-normal CDF width)')
print(f'  Timing R²    : {artifacts["timing_oof_r2"]:.4f}')
print(f'  Features     : {len(FEAT_COLS)} (V8-exact)')
print(f'  Weights      : coxnet={weights["coxnet"]:.3f}  gbm={weights["gbm"]:.3f}  rsf={weights["rsf"]:.3f}')
print()
print(f'  OOF Hybrid   : {h15:.4f}  (V8=0.9717)')
print(f'  OOF C-index  : {c15:.4f}  (V8=0.9403)')
print(f'  OOF WBS      : {wbs15:.4f}  (V8=0.0148)')
print(f'  Test close@48h: {artifacts["test_close_48h"]:.4f}')
print()
print(f'Submit: Artifacts and Submissions/submission (V15).csv')


Artifacts saved: Artifacts and Submissions/model_artifacts (V15).pkl

=== V15 Final Summary ===
  Architecture : Survival ensemble (Stage 1) + Ridge timing (Stage 2)
  Close fires  : beta=0.40*timing + 0.60*survival
  Far fires    : survival only
  Sigma        : 1.0  (log-normal CDF width)
  Timing R²    : 0.2903
  Features     : 42 (V8-exact)
  Weights      : coxnet=0.778  gbm=0.178  rsf=0.044

  OOF Hybrid   : 0.9722  (V8=0.9717)
  OOF C-index  : 0.9412  (V8=0.9403)
  OOF WBS      : 0.0145  (V8=0.0148)
  Test close@48h: 0.9615

Submit: Artifacts and Submissions/submission (V15).csv
